## Preprocessing settings
Set the paths to the raw posts and comments of one subreddit (`.zst` dump or `.csv`) and to the output folder, then run. Run once per subreddit.
`COLS` = columns kept from the raw files, everything else is dropped.

In [ ]:
import html, io, json, re
from pathlib import Path
import pandas as pd
import zstandard

# raw files and output folder (change these for each subreddit)
SUBMISSIONS_FILE = Path("path/to/Subreddit_submissions.zst")
COMMENTS_FILE    = Path("path/to/Subreddit_comments.zst")
OUT_DIR          = Path("path/to/output_folder")

# length thresholds (cleaned text)
MIN_TITLE_CHARS   = 5
MIN_COMMENT_CHARS = 25
REVIEW_N   = 25          # how many removed items to look at
CHUNK_SIZE = 200_000     # rows read at once (the comment files are big)
SEED       = 42

# columns to keep
COLS = {
    "submissions": ["created_utc", "author", "ups", "downs", "score", "url", "id",
                    "author_flair_text", "is_self", "num_comments", "selftext", "title",
                    "subreddit", "subreddit_subscribers"],
    "comments":    ["id", "link_id", "parent_id", "author", "body", "created_utc",
                    "subreddit", "score", "ups", "downs", "author_flair_text"],
}
DELETED = {"[deleted]", "[removed]"}

## Functions
`clean_text` cleans one text, `read_raw` reads a raw file in chunks, `process` runs all cleaning steps on one file.

In [ ]:
# regex replacements, applied in this order
_SUBS = [
    (r"\[([^\]]*)\]\([^)]+\)", r"\1"),          # markdown link -> link text
    (r"https?://\S+|www\.\S+", ""),              # urls
    (r"(?m)^>{1,3}\s?", ""),                    # quote marker
    (r"\*{1,3}(.*?)\*{1,3}", r"\1"),             # bold / italic
    (r"_{1,3}(.*?)_{1,3}", r"\1"),
    (r"~~(.*?)~~", r"\1"),                       # strikethrough
    (r"(?m)^#{1,6}\s+", ""),                     # headings
    (r"(?m)^(?:\*{0,2}-\s?|\*\s)", ""),        # bullets
    (r"(?m)^[-*_]{3,}\s*$", ""),                 # horizontal lines
    (r"`{1,3}(.*?)`{1,3}", r"\1"),               # code
    (r"[​‌‍﻿]", ""),         # zero-width characters
    (r"\[(deleted|removed)\]", ""),
    (r"\n{3,}", "\n\n"),
    (r"[ \t]{2,}", " "),
]
_SUBS = [(re.compile(p), r) for p, r in _SUBS]

def clean_text(t):
    # decode html entities, then run all the replacements
    t = html.unescape(t)
    for p, r in _SUBS:
        t = p.sub(r, t)
    return t.strip()


def read_raw(path, kind):
    # csv: read in chunks, only keep COLS
    if path.suffix != ".zst":
        cols = [c for c in COLS[kind] if c in pd.read_csv(path, nrows=0).columns]
        yield from pd.read_csv(path, usecols=cols, dtype=str, keep_default_na=False, chunksize=CHUNK_SIZE)
        return
    # .zst: one json record per line, collect them into chunks
    rows = []
    with open(path, "rb") as fh:
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(fh)
        for line in io.TextIOWrapper(reader, encoding="utf-8"):
            if line.strip():
                rec = json.loads(line)
                rows.append({c: "" if rec.get(c) is None else str(rec.get(c)) for c in COLS[kind]})
            if len(rows) == CHUNK_SIZE:
                yield pd.DataFrame(rows, columns=COLS[kind])
                rows = []
    if rows:
        yield pd.DataFrame(rows, columns=COLS[kind])


def process(path, kind):
    # clean one file chunk by chunk, save the cleaned rows and the too-short rows, return the counts
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    stem = path.name.split(".")[0]            # e.g. Feminism_submissions
    n = dict(original=0, deleted_removed=0, link_posts=0, too_short=0, final=0)

    for i, df in enumerate(read_raw(path, kind)):
        n["original"] += len(df)
        if kind == "submissions":
            # drop deleted / removed posts
            gone = df["selftext"].isin(DELETED)
            n["deleted_removed"] += int(gone.sum()); df = df[~gone]
            # drop link posts
            link = df["is_self"].str.lower() != "true"
            n["link_posts"] += int(link.sum()); df = df[~link].copy()
            # clean title and body, datapoint = title + body
            df["title_clean"] = df["title"].map(clean_text)
            df["selftext_clean"] = df["selftext"].map(clean_text)
            df["datapoint"] = (df["title_clean"] + "\n" + df["selftext_clean"]).str.strip()
            # title too short
            short = df["title_clean"].str.len() < MIN_TITLE_CHARS
        else:
            # drop deleted / removed comments, clean, check length
            gone = df["body"].isin(DELETED)
            n["deleted_removed"] += int(gone.sum()); df = df[~gone].copy()
            df["body_clean"] = df["body"].map(clean_text)
            short = df["body_clean"].str.len() < MIN_COMMENT_CHARS

        # save too-short rows separately (to check them below), keep the rest
        mode, header = ("w", True) if i == 0 else ("a", False)
        df[short].to_csv(OUT_DIR / f"{stem}_too_short.csv", mode=mode, header=header, index=False)
        df[~short].to_csv(OUT_DIR / f"{stem}_cleaned.csv", mode=mode, header=header, index=False)
        n["too_short"] += int(short.sum())
        n["final"] += int((~short).sum())

    print(f"{stem}: {n['original']:,} -> {n['final']:,}")
    return n

## Run
Cleans the posts and the comments (the comment file takes a while).

In [ ]:
# clean posts and comments
stats = {"submissions": process(SUBMISSIONS_FILE, "submissions"),
         "comments":    process(COMMENTS_FILE, "comments")}

## Check removed items
Random sample of items removed by the length thresholds (original text). If relevant stuff shows up here, lower the threshold and rerun.

In [ ]:
# show a random sample of the too-short items
pd.set_option("display.max_colwidth", 120)
for path, col in [(SUBMISSIONS_FILE, "title"), (COMMENTS_FILE, "body")]:
    stem = path.name.split(".")[0]
    removed = pd.read_csv(OUT_DIR / f"{stem}_too_short.csv", dtype=str, keep_default_na=False)
    print(f"\n=== {stem}: {len(removed):,} removed ===")
    display(removed[["id", col]].sample(min(REVIEW_N, len(removed)), random_state=SEED))

## Summary
Row counts per step, saved as `preprocessing_counts.csv` in the output folder.

In [ ]:
# counts per file type, and final n (posts + comments)
table = pd.DataFrame(stats).T
display(table)
table.to_csv(OUT_DIR / "preprocessing_counts.csv")
print(f"N = {table['final'].sum():,} (posts + comments)")